In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n --- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

 --- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap=100,
    separators= ["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)
print(len(chunks))

37


In [12]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print("Vector store ready")

X:\Aleena\AI projects\shopping_agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JUNCTION\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3964.15it/s]


Vector store ready


In [18]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"Chunk {i}")
    print(doc.page_content[:300])
    print()

Chunk 1
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

Chunk 2
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

Chunk 3
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)

llm = ChatGroq(
    model = "qwen/qwen3.6-27b",
    temperature=0,
    reasoning_format = "parsed",
)

chain = (
    {"context" : retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled")

RAG chain assembled


In [27]:
question = "How does international roaming work and what charges should I expect?"

print("Answer: ", chain.invoke(question))

Answer:  Based on the provided context, here is how international roaming works and what charges to expect:

**How It Works:**
- When you travel outside your home network's coverage area, your device connects to a partner network in the visited country.
- The visited network authenticates your device using an inter-operator signalling protocol (SS7 or Diameter). Your home network then validates your subscription and authorizes service.
- All data, voice, and SMS traffic is tunneled back to the home network for billing. This tunnelling process adds latency compared to local network usage.

**Charges & Roaming Zones:**
Charges are determined by the roaming zone you visit:
- **Zone A** (EU, UK, Australia, New Zealand): Lowest roaming rates
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates
- **Zone C** (Rest of World): Highest per-MB and per-minute charges

To avoid bill shock, you should purchase a roaming bundle before traveling to Zone B or C countries. Please note that if yo